# MMF Local Models
Auto-generated by mmf-agent.

In [ ]:
%pip install "mmf_sa[local] @ git+https://github.com/databricks-industry-solutions/many-model-forecasting.git@v0.1.2" --quiet

In [ ]:
%restart_python

In [ ]:
catalog = "{catalog}"
schema = "{schema}"
train_table = "{train_table}"
freq = "{freq}"
prediction_length = {prediction_length}
backtest_length = {backtest_length}
stride = {stride}
metric = "{metric}"
active_models = {active_models}
group_id = "{group_id}"
date_col = "{date_col}"
target = "{target}"

# Covariates for mmf_sa.run_forecast — see Skill 4 Feature Type Decision Guide.
# SAFE:  static_features (constant per group_id), dynamic_historical_* (past-only, for global models).
# AVOID: dynamic_future_* requires a scoring table with known future values for every future ds.
#        If missing, the pipeline errors or silently drops series. Leave as [] unless you have
#        confirmed future exogenous data and a pre-built scoring table.
static_features = {static_features}
dynamic_future_numerical = {dynamic_future_numerical}
dynamic_future_categorical = {dynamic_future_categorical}
dynamic_historical_numerical = {dynamic_historical_numerical}
dynamic_historical_categorical = {dynamic_historical_categorical}
# Scoring table: always "" (same as train_table) unless dynamic_future_* is in use (not recommended).
scoring_table = "{scoring_table}"

In [ ]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("py4j.clientserver").setLevel(logging.ERROR)
logging.getLogger("py4j.java_gateway").setLevel(logging.ERROR)

from mmf_sa import run_forecast

user = spark.sql("SELECT current_user() AS user").collect()[0]["user"]


def _exog_kwargs():
    kw = {}
    if static_features:
        kw["static_features"] = static_features
    if dynamic_future_numerical:
        kw["dynamic_future_numerical"] = dynamic_future_numerical
    if dynamic_future_categorical:
        kw["dynamic_future_categorical"] = dynamic_future_categorical
    if dynamic_historical_numerical:
        kw["dynamic_historical_numerical"] = dynamic_historical_numerical
    if dynamic_historical_categorical:
        kw["dynamic_historical_categorical"] = dynamic_historical_categorical
    return kw


train_fqn = catalog + "." + schema + "." + train_table
_st = (scoring_table or "").strip()
scoring_fqn = train_fqn if not _st else catalog + "." + schema + "." + _st

run_forecast(
    spark=spark,
    train_data=train_fqn,
    scoring_data=scoring_fqn,
    scoring_output=catalog + "." + schema + ".{use_case}_scoring_output",
    evaluation_output=catalog + "." + schema + ".{use_case}_evaluation_output",
    group_id=group_id,
    date_col=date_col,
    target=target,
    freq=freq,
    prediction_length=prediction_length,
    backtest_length=backtest_length,
    stride=stride,
    metric=metric,
    train_predict_ratio=1,
    data_quality_check=True,
    resample=False,
    active_models=active_models,
    experiment_path="/Users/" + user + "/mmf/{use_case}",
    use_case_name="{use_case}",
    **_exog_kwargs(),
)